In [ ]:
import copy

import jax
import jax.numpy as jnp
import numpy as np
from jax.nn import relu
from tqdm.autonotebook import tqdm

import adaptive_latents
from adaptive_latents import StreamingKalmanFilter, ArrayWithTime, Pipeline, proSVD, CenteringEstimator
from adaptive_latents.input_sources.autoregressor import AdamOptimizer
from adaptive_latents.regressions import BaseKNearestNeighborRegressor
from adaptive_latents.stim_regressor import StimRegressor

rng = np.random.default_rng(0)

In [ ]:
d = adaptive_latents.datasets.Odoherty21Dataset()

In [ ]:
def high_d_S(low_d_point, high_d_stim):
    return high_d_stim

def S(low_d_point, high_d_stim, pro):
    return pro.transform(high_d_S(low_d_point, high_d_stim)[None,:])

In [ ]:
def loss(s, v, lam_1=1e-3):
    u = s  # this assumes for now that the dynamics S function is an identity
    return (
            - jnp.sqrt(jnp.linalg.norm(v.T @ s))**2  # maximize dot product with the target vector
            + jnp.linalg.norm(s - v @ v.T @ s)**2  # minimize orthogonal component
            + jnp.linalg.norm(u, ord=1) * lam_1  # L1 penalty
    )

grad_loss = jax.jit(jax.value_and_grad(loss))


def design_stim(v, N=30, convergence_threshold=1e-2, max_outer_iters=20, max_inner_iters=250):
    v = np.atleast_2d(v.T).T

    s_history = []
    loss_history = []

    lam_1 = 1e-3

    for _ in range(max_outer_iters):
        s = rng.uniform(size=(130,)) * .1
        s_optimizer = AdamOptimizer(lr=0.005)

        for i in range(max_inner_iters):
            val, grad = grad_loss(s, v, lam_1=lam_1)
            s = s_optimizer.update(s,grad)
            s = relu(s)

            s_history.append(s)
            loss_history.append(val)

            if len(s_history) > 10 and jnp.linalg.norm(s_history[-2] - s_history[-1]) < convergence_threshold:
                break

        l0 = jnp.linalg.norm(s,ord=0)
        if 0 < l0 <= N:
            break
        if l0 == 0 or jnp.isnan(s).any():
            lam_1 /= 1.2
        else:
            lam_1 *= 2
    return np.array(s / s.max())


In [ ]:
def do_experiment(input_arrays, decay_rate=.9, stim_scale=1, stim_p=0.05, rng=None, s_inputs_to_evaluate_on=None):
    if rng is None:
        rng = np.random.default_rng(0)

    centerer = CenteringEstimator()

    pro = proSVD(k=10)

    sr = StimRegressor(
        autoreg=StreamingKalmanFilter(),
        stim_reg=BaseKNearestNeighborRegressor(k=1, maxlen=100),
        attempt_correction=True
    )

    stims = []
    predictions = []
    latents = []
    dt_X = []
    s_hat_evals = []
    s_eval = []

    s_inputs_evaluated_on = []

    to_add = np.zeros(input_arrays[0].shape[1])

    for input_array in input_arrays:
        pbar = tqdm(total=round(input_array.t[-1],2))
        for data in Pipeline().streaming_run_on(input_array):

            latent_location = pro.transform(centerer.transform(data))

            stim = np.zeros(data.shape[1])
            if rng.random() < stim_p and pro.is_initialized:
                stim = design_stim(pro.Q[:,0]) * stim_scale
                to_add += high_d_S(latent_location, stim)

            if s_inputs_to_evaluate_on == 'record' and np.any(stim):
                s_inputs_evaluated_on.append(copy.deepcopy((latent_location, stim, pro)))
                s_eval.append(S(latent_location, stim, pro))

            data = data + to_add
            to_add = to_add * decay_rate

            data = centerer.step(data)
            data = pro.step(data)
            latents.append(data)

            qX = ArrayWithTime([[1]], data.t)
            sr.step(np.array([[stim]]), stream='stim')
            prediction = sr.step(qX, stream='dt_X')
            sr.step(data, stream='X')



            stims.append(ArrayWithTime(stim, data.t))
            predictions.append(prediction)
            dt_X.append(qX)

            if not isinstance(s_inputs_to_evaluate_on, str) and s_inputs_to_evaluate_on is not None and np.any(stim):
                point_evals = []
                for latent_location, stim, _ in s_inputs_to_evaluate_on:
                    stim_reg_input = np.hstack([latent_location.flatten(), stim.flatten()])
                    diff = sr.stim_reg.predict(stim_reg_input)
                    point_evals.append(diff.flatten())
                s_hat_evals.append(ArrayWithTime(point_evals,data.t))

                if not s_eval:
                    for inputs in s_inputs_to_evaluate_on:
                        s_eval.append(S(*inputs))


            if not sr.autoreg.get_parameter_fitting_state():
                sr.autoreg.set_parameter_fitting_state(True)

            pbar.update(round(data.t,2) - pbar.n)
        sr.autoreg.set_parameter_fitting_state(False)


    stims = ArrayWithTime.from_list(stims)
    predictions = ArrayWithTime.from_list(predictions, drop_early_nans=True, squeeze_type='to_2d')
    latents = ArrayWithTime.from_list(latents, squeeze_type='to_2d')
    s_hat_evals = ArrayWithTime.from_list(s_hat_evals, drop_early_nans=True)
    s_eval = np.squeeze(s_eval)
    return (predictions, latents, stims), (s_hat_evals, s_eval, s_inputs_evaluated_on)


In [ ]:
stim_scale = 20
(predictions, latents, stims), _ = do_experiment([d.neural_data], stim_p=0.005, stim_scale=stim_scale)


In [ ]:
predictions.shape, latents.shape